# W1 보조 — 작은 예시 모음

본 노트북은 본 W1 자료를 보기 전 또는 본 후에, 핵심 개념을 더 작은 예시로 확인하는 mini 예제 모음입니다. 한 셀씩 실행하며 결과를 눈으로 익히세요.

각 예시는 독립적이라 순서대로 실행할 필요는 없습니다.

## 0. 환경 준비

본 노트북은 본 자료의 `helpers/dr_utils.py` 를 그대로 사용합니다. 본 노트북을 `COMMON/shared/week1/notebooks/` 옆에 두고 실행하거나, 경로를 본인 환경에 맞게 수정하세요.

In [ ]:
import sys
from pathlib import Path
# 본 자료 helpers 경로 (필요에 따라 수정)
sys.path.insert(0, str(Path('../../COMMON/shared/week1/helpers').resolve()))

import numpy as np
import matplotlib.pyplot as plt
from dr_utils import (load_volume, porosity, normalize_to_float,
                      otsu_threshold, binarize_otsu,
                      linear_interpolate_slice,
                      predict_linear_k, predict_cubic_k,
                      neighbor_targets, eval_targets, ssim_3d_mean,
                      setup_plot_style, ORANGE, NAVY, GREEN)
setup_plot_style()

DATA = Path('../../COMMON/shared/week1/data').resolve()
bb = load_volume(DATA / 'BB_256.bin')
print(f'BB shape={bb.shape}, φ={porosity(bb)*100:.2f}%')

## 예시 1 — 한 슬라이스만 자유롭게 보기

본 자료의 `show_three_axis`는 세 축을 한 번에 보여줍니다. 더 단순히 한 슬라이스만 보고 싶을 때:

In [ ]:
# z 위치를 자유롭게 바꿔보세요
z = 100

plt.figure(figsize=(5, 5))
plt.imshow(bb[z], cmap='gray')
plt.title(f'BB · z={z} · φ={bb[z].mean()*100:.1f}%')
plt.axis('off')
plt.show()

**시도**: `z` 를 0, 50, 128, 200, 255 로 바꿔보고 슬라이스마다 패턴이 어떻게 다른지 관찰.

## 예시 2 — α 값에 따른 단순 보간 변화

양옆 이웃 슬라이스 사이를 α=0~1로 변화시키며 결과를 봅니다. (이웃 거리 k일 때 가운데는 α=0.5)

In [ ]:
z_before, z_after = 60, 66   # 양옆 이웃 슬라이스 위치 (t±k)

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, len(alphas), figsize=(13, 3))
for ax, a in zip(axes, alphas):
    pred = linear_interpolate_slice(bb[z_before], bb[z_after], a)
    ax.imshow(pred, vmin=0, vmax=1, cmap='gray')
    ax.set_title(f'α={a:.2f}')
    ax.axis('off')
plt.suptitle(f'z={z_before} ↔ z={z_after} 이웃 사이의 선형 보간', y=1.05)
plt.tight_layout()
plt.show()

**시도**:
- `z_before, z_after` 간격을 1 → 5 → 15 → 30 으로 늘리면 α=0.5 결과가 얼마나 흐려지는지
- α=0.5 일 때 결과 부피의 공극률을 출력 (`pred.mean()`) 해 보고 양 끝 슬라이스의 평균과 비교

## 예시 3 — 이웃 거리 k 변화의 영향

k=1, 2, 3, 5 네 가지 이웃 거리에서 Linear 예측 결과 시각화.

In [ ]:
z_show = 62
fig, axes = plt.subplots(1, 5, figsize=(15, 4))
axes[0].imshow(bb[z_show], cmap='gray'); axes[0].set_title(f'원본 z={z_show}')
for ax, k in zip(axes[1:], [1, 2, 3, 5]):
    rec = predict_linear_k(bb, k)
    ax.imshow(rec[z_show], cmap='gray')
    ax.set_title(f'Linear k={k}\n|Δφ|={eval_targets(rec, bb, k)["dphi_pp"]:.2f}%p')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

**관찰**: 이웃 거리 k가 클수록 슬라이스가 "부드럽게" 변하는데, 이는 원본의 세부 구조를 잃어버린다는 의미입니다. |Δφ| 만으로는 안 보이는 차이가 시각으로 드러납니다.

## 예시 4 — Otsu 에 노이즈를 더해보기

본 binary 데이터에 노이즈를 더해 "가짜 grayscale" 을 만들고 Otsu 동작 확인.

In [ ]:
rng = np.random.default_rng(42)

for sigma in [0.05, 0.2, 0.4]:
    gray = bb[128].astype(np.float32) + rng.normal(0, sigma, bb[128].shape)
    gray = np.clip(gray, 0, 1)
    t = otsu_threshold(gray)
    bin_, _ = binarize_otsu(gray)
    err = abs(bin_.mean() - bb[128].mean()) * 100
    print(f'σ={sigma:.2f} → Otsu t={t:.3f} | binarize φ={bin_.mean()*100:.2f}% | |Δφ| vs 원본 = {err:.2f}%p')

**관찰**: σ가 작을 때는 Otsu가 정확하지만, σ가 커지면 두 봉우리가 합쳐져 Otsu의 임계값이 흔들립니다.

**시도**: σ 를 0.5, 0.8 까지 늘려보고 어디부터 Otsu 결과가 의미를 잃는지 확인.

## 다음 단계

예시들이 익숙해졌으면, 본 자료의 노트북 `W1_load_and_explore.ipynb` 에서 같은 흐름을 더 깊이 다뤄봅니다.